In [0]:
import os

CATALOG_NAME = 'workspace'
SCHEMA_NAME = 'lab_2026'
VOLUME_DIR = os.path.abspath('/Volumes/{}/{}/'.format(CATALOG_NAME, SCHEMA_NAME))
RAW_DIR = os.path.join(VOLUME_DIR, 'raw')
COMPRESSED_RAW_DIR = os.path.join(RAW_DIR, '.compressed')
if not os.path.exists(COMPRESSED_RAW_DIR):
    raise Exception(f"{COMPRESSED_RAW_DIR} Not Found!")

INPUT_FILE = 'online_retail_II.csv'
compressed_file_path = os.path.join(COMPRESSED_RAW_DIR, INPUT_FILE + '.zip')
dbutils.fs.ls(compressed_file_path)

[FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/.compressed/online_retail_II.csv.zip', name='online_retail_II.csv.zip', size=15217139, modificationTime=1766781209000)]

In [0]:
import zipfile

csv_file_path = os.path.join(RAW_DIR)
with zipfile.ZipFile(compressed_file_path, "r") as zip_ref:
    zip_ref.extractall(csv_file_path)
dbutils.fs.ls(csv_file_path)

[FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/', name='.checkpoints/', size=0, modificationTime=1772520590796),
 FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/.compressed/', name='.compressed/', size=0, modificationTime=1772520590796),
 FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv', name='online_retail_II.csv', size=94850204, modificationTime=1772520590000)]

In [0]:
spark.read.csv(RAW_DIR, header=True, inferSchema=False).limit(5).display()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
bronze_schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("Price", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True)
])
checkpoint_location = os.path.join(RAW_DIR, '.checkpoints', 'stream_online_retail')

In [0]:
dbutils.fs.ls('dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/')
# dbutils.fs.rm('dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/', True)

[FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/online_retail/', name='online_retail/', size=0, modificationTime=1772520592793),
 FileInfo(path='dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/stream_online_retail/', name='stream_online_retail/', size=0, modificationTime=1772520592793)]

In [0]:
source_df = (
    spark.readStream
    .format("csv")
    .schema(bronze_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .option("checkpointLocation", checkpoint_location)
    .load(RAW_DIR)
)

import re
renamed_columns = dict()
for col in source_df.columns:
    renamed_columns[col] = re.sub(r'[^a-zA-Z0-9]', '_', col.lower())

from pyspark.sql import functions as F
raw_df = (
    source_df
    .withColumn('_hash_md5', F.md5(F.concat_ws(',', *source_df.columns)))
    .withColumn('_ingest_timestamp', F.current_timestamp())
    .withColumn('_ingest_author', F.current_user())
    .withColumn('_source_file', F.col("_metadata.file_path"))
    .withColumnsRenamed(renamed_columns)
)

In [0]:
query = (
    raw_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .table("lab_2026.stream_online_retail")
)

In [0]:
%sql

select * from lab_2026.stream_online_retail
order by `_ingest_timestamp` desc
limit 20

invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,_hash_md5,_ingest_timestamp,_ingest_author,_source_file
489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,f7a1f3e1eb542ef39cf2b255a4c4c0a3,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,8f540e2dceb601d1c14994732a5dec97,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,e3aad9c7dac0fee528d0978f4d1d4ad9,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,5ff842f39a6d8bdb90aa2522efa4072f,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,61121fde713280adf87973245df2e166,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489435,22353,LUNCHBOX WITH CUTLERY FAIRY CAKES,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,ad09515878510b407c1ac1a3b93ff7cd,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489436,22119,PEACE WOODEN BLOCK LETTERS,3,2009-12-01 09:06:00,6.95,13078.0,United Kingdom,3692cf19366ad32c3bca00c73cd5e696,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,b36f5ca093e3e5221da90ba56d85ee86,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,cf37b3e9773be28923bda182977ce0d9,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01 09:06:00,1.69,13078.0,United Kingdom,867c28b0a2206e3965ba28ebc5c5f18e,2026-03-03T06:46:33.373Z,gopinadh5g7@sasi.ac.in,dbfs:/Volumes/workspace/lab_2026/raw/online_retail_II.csv
